In [1]:
import spacy
from spacy import displacy
from spacy.tokens import Doc, Span, Token

from biz.dfch.asdste100vocab import Vocab
from biz.dfch.asdste100vocab import Word
from biz.dfch.asdste100vocab import WordStatus
from biz.dfch.asdste100vocab import WordType

from biz.dfch.ste100parser import GrammarType
from biz.dfch.ste100parser import Inspector
from biz.dfch.ste100parser import Parser
from biz.dfch.ste100parser import ParserAction
from biz.dfch.ste100parser import Ste100Doc
from biz.dfch.ste100parser.serializer.text_interpreter import TextInterpreter

from biz.dfch.ste100parser.serializer.token_base import TokenBase
from biz.dfch.ste100parser.serializer.token_base import ListToken

from biz.dfch.ste100parser.token_registry import TokenRegistry
from biz.dfch.ste100parser.rule_registry import RuleRegistry
from biz.dfch.ste100parser.rule_registry import RuleContext

nlp = spacy.load("en_core_web_sm")


In [ ]:
text = """The button was pressed 2 times. The end user presses the button two times. The pressed button is orange."""
text = """During transmission, the data was corrupted.
During transmission, something corrupted the data.
Transmission corrupted the data.
The safety procedures are given by the manufacturer.
The main gear leg is held by the side stay.
  * The volume control can be adjusted.

A) The valve will be adjusted during the test.
  * The oil temperature must be adjusted before the start of the test.

"""

parser = Parser(GrammarType.ASD_STE100_9)
tree = parser.invoke(text, action=ParserAction.PASS2)
vocab = Vocab()

interpreter = TextInterpreter(vocab=vocab)
tokens = interpreter.invoke(tree)
doc = Ste100Doc(tokens)
inspector = Inspector()
structure = inspector.ste100doc(doc)
print(structure)

token_registry = TokenRegistry.Factory.get_instance()
rule_context = RuleContext(vocab, token_registry)
rule_registry = RuleRegistry()
rule_registry.install_rules("biz.dfch.ste100parser.rule_repository")

def _process_tokens(tokens: list[TokenBase]):
    for token in tokens:
        if isinstance(token, ListToken):
            _process_tokens(token.tokens)
        # print(f"[{type(token).__name__}] Processing token '{token.text}' ...")
        rules = rule_registry.get_rules(token)
        for rule in rules:
            # print(f"Processing rule '{rule.rule_id}' [{rule.priority}] [{type(token).__name__}] ...")
            try:
                test_results = rule.examine(token, rule_context)
                for test_result in test_results:
                    print(f"[{test_result.severity}] {test_result.rule_id}: '{test_result.message}'")
            except Exception as ex:
                print(f"'{type(rule).__name__}' failed [{ex}]")

_process_tokens(list(doc))

# displacy.render(doc, jupyter=True)


In [4]:
text = """Close the door.
You close the door.
The door is closed.
Turn off the engine.
Do not enter.

A) Close the door.
B) You close the door.
C) The door is closed.
D) Turn off the engine.
E) Do not enter.

"""

parser = Parser(GrammarType.ASD_STE100_9)
tree = parser.invoke(text, action=ParserAction.PASS2)
vocab = Vocab()

interpreter = TextInterpreter(vocab=vocab)
tokens = interpreter.invoke(tree)
doc = Ste100Doc(tokens)
inspector = Inspector()
structure = inspector.ste100doc(doc)
print(structure)

token_registry = TokenRegistry.Factory.get_instance()
rule_context = RuleContext(vocab, token_registry)
rule_registry = RuleRegistry()
rule_registry.install_rules("biz.dfch.ste100parser.rule_repository")

def _process_tokens(tokens: list[TokenBase]):
    for token in tokens:
        if isinstance(token, ListToken):
            _process_tokens(token.tokens)
        # print(f"[{type(token).__name__}] Processing token '{token.text}' ...")
        rules = rule_registry.get_rules(token)
        for rule in rules:
            # print(f"Processing rule '{rule.rule_id}' [{rule.priority}] [{type(token).__name__}] ...")
            try:
                test_results = rule.examine(token, rule_context)
                for test_result in test_results:
                    print(f"[{test_result.severity}] {test_result.rule_id}: '{test_result.message}'")
            except Exception as ex:
                print(f"'{type(rule).__name__}' failed [{ex}]")

_process_tokens(list(doc))

# displacy.render(doc, jupyter=True)


#5 [TEXT]: 'Close'.
#1 [WS]: ' '.
#3 [TEXT]: 'the'.
#1 [WS]: ' '.
#5 [TEXT]: 'door.'.
#1 [LINEBREAK]: '
'.
#3 [TEXT]: 'You'.
#1 [WS]: ' '.
#5 [TEXT]: 'close'.
#1 [WS]: ' '.
#3 [TEXT]: 'the'.
#1 [WS]: ' '.
#5 [TEXT]: 'door.'.
#1 [LINEBREAK]: '
'.
#3 [TEXT]: 'The'.
#1 [WS]: ' '.
#4 [TEXT]: 'door'.
#1 [WS]: ' '.
#2 [TEXT]: 'is'.
#1 [WS]: ' '.
#7 [TEXT]: 'closed.'.
#1 [LINEBREAK]: '
'.
#4 [TEXT]: 'Turn'.
#1 [WS]: ' '.
#3 [TEXT]: 'off'.
#1 [WS]: ' '.
#3 [TEXT]: 'the'.
#1 [WS]: ' '.
#7 [TEXT]: 'engine.'.
#1 [LINEBREAK]: '
'.
#2 [TEXT]: 'Do'.
#1 [WS]: ' '.
#3 [TEXT]: 'not'.
#1 [WS]: ' '.
#6 [TEXT]: 'enter.'.
#35 [paragraph]: '[Tree('TEXT', ['Close']), Tree('WS', ['1']), Tree('TEXT', ['the']), Tree('WS', ['1']), Tree('TEXT', ['door.']), Tree('LINEBREAK', ['\n']), Tree('TEXT', ['You']), Tree('WS', ['1']), Tree('TEXT', ['close']), Tree('WS', ['1']), Tree('TEXT', ['the']), Tree('WS', ['1']), Tree('TEXT', ['door.']), Tree('LINEBREAK', ['\n']), Tree('TEXT', ['The']), Tree('WS', ['1']), Tree('TEXT',

'You' [PRON] [nsubj]
'close' [VERB] [ROOT]
'the' [DET] [det]
'door' [NOUN] [dobj]
'.' [PUNCT] [punct]


'The' [DET] [det]
'door' [NOUN] [nsubjpass]
'is' [AUX] [auxpass]
'closed' [VERB] [ROOT]
'.' [PUNCT] [punct]


[WARNING] R3.6: 'Descriptive: [['door', 'is', 'closed']] Use the active voice. In descriptive writing, you can use the passive voice only when the agent is unknown.'
'Turn' [VERB] [ROOT]
'off' [ADP] [prt]
'the' [DET] [det]
'engine' [NOUN] [dobj]
'.' [PUNCT] [punct]


'Do' [AUX] [aux]
'not' [PART] [neg]
'enter' [VERB] [ROOT]
'.' [PUNCT] [punct]


...

'UseImperativeForm' failed [<class 'NoneType'>]
'You' [PRON] [nsubj]
'close' [VERB] [ROOT]
'the' [DET] [det]
'door' [NOUN] [dobj]
'.' [PUNCT] [punct]


'UseImperativeForm' failed [<class 'NoneType'>]
'The' [DET] [det]
'door' [NOUN] [nsubjpass]
'is' [AUX] [auxpass]
'closed' [VERB] [ROOT]
'.' [PUNCT] [punct]


[ERROR] R3.6: 'Procedural: [['door', 'is', 'closed']] Use the active voice. In descriptive writing, you can use the passive voice only when the agent is unknown.'
'UseImperativeForm' failed [<class 'NoneType'>]
'Turn' [VERB] [ROOT]
'off' [ADP] [prt]
'the' [DET] [det]
'engine' [NOUN] [dobj]
'.' [PUNCT] [punct]


'UseImperativeForm' failed [<class 'NoneType'>]
'Do' [AUX] [aux]
'not' [PART] [neg]
'enter' [VERB] [ROOT]
'.' [PUNCT] [punct]


'UseImperativeForm' failed [<class 'NoneType'>]
